This notebook compiles and organizes the behavioral trial data. Access to 
OpenMind is required.

Last updated on January 28, 2025, by Ruimin Gao.

In [1]:
import os
import pandas as pd
import numpy as np
import re
import pathlib
import scipy.io

cwd = pathlib.Path('~/DiffTasks/behavioral_analysis_2025/qc/')

In [ ]:
# Read all raw Data
data = []

raw_data_folder = 'Data'
table_files = [f for f in os.listdir(raw_data_folder) if f.endswith('.csv')]
for name in table_files:
    session = name.replace('.csv', '').replace('langloc_DiffTasks_', '')
    session = re.sub(r"_run\d+", '', session)
    if 'FED_' not in session: session_ = session.replace('FED', 'FED_')
    else: session_ = session
    if len(session_) != 17: continue

    # Read the trial results
    # Run, Version, Set, Trial, Condition, Onset, Sentence, Probe_or_Question, Response, RT
    table = pd.read_csv(os.path.join(raw_data_folder, name))

    # Add Subject and Session
    # Find $HOME/evlab_subjects/filename where filename contains session
    subject_folder = os.path.join(os.environ['HOME'], 'evlab_subjects')
    subject_file = [f for f in os.listdir(subject_folder) if session_ in f]
    if len(subject_file) == 0:
        print(f'No subject file found for {session_}')
        subject = 'NA'
    else:
        subject = subject_file[0].split('_')[0]
    table['Subject'] = subject
    table['Session'] = session

    table['Task'] = table.apply(
        lambda row: 'ButtonPress' if row['Version'] == 1 or (row['Version'] == 6 and row['Condition'] == 'S')
        else 'MemoryAll' if row['Version'] == 2 or (row['Version'] in [4,5,6] and row['Condition'] == 'N')
        else 'MemoryLast' if row['Version'] == 3 
        else 'ComprehensionQ' if row['Version'] == 4
        else 'Sentiment',
        axis=1
    )

    table['Sentence_Type'] = table.apply(
        lambda row: 'TwoSentences' if row['Version'] in [4,5,6] and row['Condition'] == 'S' else 'OneSentence',
        axis=1
    )
    
    # Grade
    # For ButtonPress, correct if any response,
    # For others, correct if response matches the probe/question (if given)
    table['Response_Rate'] = table['Response'].apply(lambda x: 1 if not (pd.isnull(x)) else 0)
    table['Accuracy_Strict'] = table.apply(
        lambda row: not pd.isnull(row['Response']) if row['Task'] == 'ButtonPress'
        else np.nan if pd.isnull(row['Probe_or_Question']) 
        else row['Response'] == row['Probe_or_Question'],
        axis=1
    )
    # Lax Accuracy: For ButtonPress, same as Strict Accuracy. 
    # For others, does not count if no response was given.
    table['Accuracy_Lax'] = table.apply(
        lambda row: row['Accuracy_Strict'] if row['Task'] == 'ButtonPress'
        else row['Accuracy_Strict'] if not pd.isnull(row['Response'])
        else np.nan,
        axis=1
    )
   
    data.append(table[['Subject', 'Session', 'Version', 'Condition', 
        'Set', 'Run', 'Trial', 'Onset', 'Task', 
        'Sentence', 'Sentence_Type', 
        'Probe_or_Question', 'Response', 'RT', 
        'Response_Rate', 'Accuracy_Strict', 'Accuracy_Lax']])

data = pd.concat(data, ignore_index=True)

In [ ]:
data.to_csv(cwd / 'Data' / 'processed_data_by_trial.csv', index=False)

Inclusion / Exclusion:

In [43]:
cwd = pathlib.Path('~/Documents/DiffTasks/behavioral_analysis_2025/')
data = pd.read_csv(cwd / 'qc' / 'Data' / 'processed_data_by_trial.csv')

In [44]:
# Handling of duplicate experiments
# Remove FED_20180720a_3T2 Version 5
# Remove FED_20190131c_3T2 Version 1-4
data = data[~((data['Session'] == 'FED_20180720a_3T2') & (data['Version'] == 5))]
data = data[~((data['Session'] == 'FED_20190131c_3T2') & (data['Version'] < 5))]

In [45]:
# number of unique combination of subject, session, run
count = data.groupby(['Subject', 'Session', 'Run']).size().reset_index(name='Counts')
print(f'Number of unique subject-session-run combinations: {len(count)}')

Number of unique subject-session-run combinations: 522


In [46]:
# Trials to exclude due to bad data
to_exclude = [
    'FED_20190117d_3T2',        # 708 sleepy
    'FED_20170324a_3T2',        # 430 bad ART
    'FED_20190208c_3T2',        # 721 bad ART
    'FED_20190214a_3T2',        # 723 bad ART
    'FED_20190117b_3T2',        # 706 only one run for all tasks
]
data = data[~data['Session'].isin(to_exclude)]

In [47]:
# number of unique combination of subject, session, run
count = data.groupby(['Subject', 'Session', 'Run']).size().reset_index(name='Counts')
print(f'Number of unique subject-session-run combinations: {len(count)}')

Number of unique subject-session-run combinations: 493


In [48]:
# Fix responses missed for some conditions
data_grouped = data.groupby(['Subject', 'Version', 'Run', 'Condition'])['Response_Rate'].mean().reset_index()
to_fix = data_grouped[data_grouped['Response_Rate'] == 0][['Subject', 'Version', 'Run', 'Condition']].apply(tuple, axis=1).unique().tolist()
data['Response_Rate'] = data.apply(
    lambda row: np.nan if (row['Subject'], row['Version'], row['Run'], row['Condition']) in to_fix
    else row['Response_Rate'],
    axis=1
)

In [49]:
response_rate = data.groupby(['Session', 'Version', 'Run'])['Response_Rate'].mean()
mean_ = response_rate.mean()
sd_ = response_rate.std()
lo_bound = mean_ - 3 * sd_
lo_bound

np.float64(0.6305809500986447)

In [53]:
excluded = data.groupby(['Session', 'Version', 'Run']).filter(
    lambda x: x['Response_Rate'].mean() < 0.631 
)
# print combination of Subject, Version, Run
print('Excluded combinations due to low response rate:')
for _, row in excluded[['Subject', 'Version', 'Run']].drop_duplicates().iterrows():
    print(f'Subject: {row["Subject"]}, Version: {row["Version"]}, Run: {row["Run"]}')

Excluded combinations due to low response rate:
Subject: 866, Version: 5, Run: 1
Subject: 551, Version: 3, Run: 7
Subject: 551, Version: 3, Run: 8
Subject: 602, Version: 1, Run: 2
Subject: 602, Version: 4, Run: 6
Subject: 602, Version: 2, Run: 7
Subject: 602, Version: 2, Run: 8
Subject: 714, Version: 3, Run: 9
Subject: 716, Version: 3, Run: 3
Subject: 716, Version: 1, Run: 6
Subject: 719, Version: 2, Run: 1
Subject: 679, Version: 4, Run: 4
Subject: 679, Version: 1, Run: 6
Subject: 679, Version: 2, Run: 7
Subject: 843, Version: 3, Run: 3
Subject: 848, Version: 4, Run: 6
Subject: 848, Version: 3, Run: 7


In [17]:
included = data.groupby(['Session', 'Version', 'Run']).filter(
    lambda x: x['Response_Rate'].mean() >= lo_bound 
)
included_tuples = included[['Session', 'Version', 'Run']].apply(tuple, axis=1).unique().tolist()
included_data_by_trial = data[data.apply(lambda x: (x['Session'], x['Version'], x['Run']) in included_tuples, axis=1)]

# Remove the trial if there is only one 'Run' associated with 'Session', 'Version', 'Condition'
lonely_runs = included.groupby(['Session', 'Version'])['Run'].nunique()
lonely_runs = lonely_runs[lonely_runs == 1]
lonely_runs = lonely_runs.index.tolist()
included_data_by_trial = included_data_by_trial[~included_data_by_trial.apply(lambda x: (x['Session'], x['Version']) in lonely_runs, axis=1)]

In [21]:
# number of unique combination of subject, session, run
count = included_data_by_trial.groupby(['Subject', 'Session', 'Run']).size().reset_index(name='Counts')
print(f'Number of unique subject-session-run combinations: {len(count)}')

Number of unique subject-session-run combinations: 456


In [23]:
# count number of unique subhject in included_data_by_trial for each Version
unique_subjects_per_version = included_data_by_trial.groupby('Version')['Subject'].nunique()
print('Number of unique subjects per Version:')
print(unique_subjects_per_version)

Number of unique subjects per Version:
Version
1    49
2    38
3    36
4    48
5    38
6    19
Name: Subject, dtype: int64


In [20]:
# count of unique 'Subject' for each ['Version', 'Condition']
# included_data_by_trial.groupby(['Version', 'Condition'])['Subject'].nunique()

In [ ]:
# Number of subjects who has all Version=1,2,3
subj123 = included_data_by_trial.groupby('Subject')['Version'].unique().apply(lambda x: {1,2,3}.issubset(set(x)))
subj123 = subj123[subj123].index.tolist()

subj1234 = included_data_by_trial.groupby('Subject')['Version'].unique().apply(lambda x: {1,2,3,4}.issubset(set(x)))
subj1234 = subj1234[subj1234].index.tolist()

subj1235 = included_data_by_trial.groupby('Subject')['Version'].unique().apply(lambda x:{1,2,3,5}.issubset(set(x)))
subj1235 = subj1235[subj1235].index.tolist()

subj1236 = included_data_by_trial.groupby('Subject')['Version'].unique().apply(lambda x:{1,2,3,6}.issubset(set(x)))
subj1236 = subj1236[subj1236].index.tolist()

print(f'Number of subjects who have all Version 1,2,3: {len(subj123)}')
print(f'Number of subjects who have all Version 1,2,3,4: {len(subj1234)}')
print(f'Number of subjects who have all Version 1,2,3,5: {len(subj1235)}')
print(f'Number of subjects who have all Version 1,2,3,6: {len(subj1236)}')

In [ ]:
included_data_by_trial.to_csv(cwd / 'Data' / 'included_data_by_trial.csv', index=False)
data.to_csv(cwd / 'Data' / 'all_data_by_trial.csv', index=False)

Examine whether these subjects have taken the spatialFIN task.

In [ ]:
included_data_by_trial = pd.read_csv(cwd / 'Data' / 'included_data_by_trial.csv')

subjects = included_data_by_trial['Subject'].unique()
subjects

In [ ]:
spatialFIN_sessions = []

subject_folder = os.path.join(os.environ['HOME'], 'evlab_subjects')
for subject in subjects:
    subject_files = [f for f in os.listdir(subject_folder) 
        if f.startswith(str(subject)) and
        f.endswith('PL2017') and
        os.path.isdir(os.path.join(subject_folder, f)) and 
        'firstlevel_spatialFIN' in os.listdir(os.path.join(subject_folder, f))]

    for spatialFIN_file in subject_files:
        try:
            spatialFIN_session = spatialFIN_file.replace(str(subject)+'_', '').replace('_PL2017', '')
            
            mat_file = os.path.join(subject_folder, spatialFIN_file, 'firstlevel_spatialFIN', 'modelspecification.mat')
            mat_data = scipy.io.loadmat(mat_file, struct_as_record=True, squeeze_me=False)

            matlabbatch = mat_data.get('matlabbatch')
            runs = matlabbatch.item()[0].item()[0].item()[0].item()[0].item()[4].size
            spatialFIN_sessions.append((subject, spatialFIN_session, runs))
        except Exception as e:
            print(e)
            continue

spatialFIN_sessions = pd.DataFrame(spatialFIN_sessions, columns=['Subject', 'Session', 'Runs'])

In [ ]:
spatialFIN_sessions.to_csv(cwd / 'spatialFIN_sessions.csv', index=False)

To generate the session sheet for toolbox analyses.

In [ ]:
import os
import pandas as pd
import numpy as np
import re
import pathlib
cwd = pathlib.Path('/Users/rgao76/Documents/DiffTasks/behavioral_analysis_2025/qc/')

In [ ]:
data = pd.read_csv(cwd / 'Data' / 'included_data_by_trial.csv')
info = data[['Subject', 'Version', 'Session']].drop_duplicates()
info['Session'] = info['Session'].apply(lambda x: x if 'FED_' in x else x.replace('FED', 'FED_'))
info = info.pivot(index='Subject', columns='Version', values='Session').reset_index()

In [ ]:
all_data = pd.read_csv(cwd / 'Data' / 'all_data_by_trial.csv')
info_all = all_data[['Subject', 'Version', 'Session']].drop_duplicates()
info_all['Session'] = info_all['Session'].apply(lambda x: x if 'FED_' in x else x.replace('FED', 'FED_'))
info_all = info_all.pivot(index='Subject', columns='Version', values='Session').reset_index()

In [ ]:
spatialFIN_sessions = pd.read_csv(cwd / 'spatialFIN_sessions.csv')
spatialFIN_sessions = spatialFIN_sessions.groupby('Subject').last().reset_index()
spatialFIN_sessions.to_csv(cwd / 'spatialFIN_sessions_selected.csv', index=False)
spatialFIN_sessions['Runs'].value_counts()

In [ ]:
# Merge spatialFIN_sessions to info by Subject
info = info.merge(spatialFIN_sessions, on='Subject', how='outer')
info = info.drop(columns=['Runs']).rename(columns={
    'Subject': 'UID',
    'Session': 'spatialFIN',
    1: 'langloc_DiffTasks_1', 
    2: 'langloc_DiffTasks_2',
    3: 'langloc_DiffTasks_3',
    4: 'langloc_DiffTasks_4',
    5: 'langloc_DiffTasks_5',
    6: 'langloc_DiffTasks_6',
}).fillna('NA')
info.to_csv(cwd / 'sessions4toolbox.csv', index=False)

In [ ]:
info_all = info_all.merge(spatialFIN_sessions, on='Subject', how='outer')
info_all = info_all.drop(columns=['Runs']).rename(columns={
    'Subject': 'UID',   
    'Session': 'spatialFIN',
    1: 'langloc_DiffTasks_1',
    2: 'langloc_DiffTasks_2',
    3: 'langloc_DiffTasks_3',
    4: 'langloc_DiffTasks_4',
    5: 'langloc_DiffTasks_5',
    6: 'langloc_DiffTasks_6',
}).fillna('NA')

In [ ]:
info_all = info_all.where(info != info_all, 'NA')
info_all = info_all.where(info_all == 'NA', 'EX:' + info_all.astype(str))
info_all = info_all.where(info_all != 'NA',  info.astype(str))
info_all.to_csv(cwd / 'sessions_all.csv', index=False)

From now on, `sessions4toolbox.csv` contains sessions post QC with manual inspection:

In [ ]:
import pandas as pd

sessions4toolbox = pd.read_csv('sessions4toolbox.csv')
non_nan_counts = sessions4toolbox.drop(columns=['UID']).count()
print(non_nan_counts)

import pathlib
cwd = pathlib.Path('/Users/rgao76/Documents/DiffTasks/behavioral_analysis_2025/qc/')

In [ ]:
# the list of tasks completed
counter = {} # for each combination of tasks, count the number of subjects
counter_sep = {} # V1 to V6, count nonNaN subject number

for v in range(1, 7):
    counter_sep[v] = sessions4toolbox[f'langloc_DiffTasks_{v}'].notna().sum()

for s in sessions4toolbox['UID']:
    tasks = sessions4toolbox[sessions4toolbox['UID'] == s].iloc[:, 1:7].dropna(axis=1).columns.tolist()
    tasks = tuple([int(task[-1]) for task in sorted(tasks)])
    counter[tasks] = counter.get(tasks, 0) + 1

df_counter = pd.DataFrame(columns=['V1', 'V2', 'V3', 'V4', 'V5', 'V6', '# Participants'])
for k, v in counter.items():
    row = [''] * 6
    for i, task in enumerate(k):
        row[task-1] = 'X'
    row.append(v)
    df_counter.loc[len(df_counter)] = row
df_counter.loc[len(df_counter)] = [counter_sep.get(i, 0) for i in range(1, 7)] + ['']
df_counter.to_csv(cwd /'counter.csv', index=False)

In [ ]:
data = pd.read_csv(cwd / 'Data' / 'included_data_by_trial.csv')
# re-filter but the current sessions4toolbox.csv, i.e., only keep if sessions4toolbox[UID==row['Subject'] & langloc_DiffTasks_{row['Version']}] is not 'NA'
def filter_sessions(row):
    return not pd.isna(sessions4toolbox.loc[
        (sessions4toolbox['UID'] == row['Subject']) &
        (sessions4toolbox[f'langloc_DiffTasks_{row["Version"]}'] != 'NA')
    ]).empty
included_data_by_trial = data[data.apply(filter_sessions, axis=1)]
included_data_by_trial

In [ ]:
included_data_by_trial.to_csv(cwd / 'Data' / 'included_data_by_trial_postQC.csv', index=False)